# 🔥 Grad-CAM: 모델은 그림의 어디를 보고 판단할까?
작은 CNN 에게 **원(0) / 네모(1)** 를 구분하도록 학습시킨 뒤,
- 층마다 **특징 맵**이 어떻게 다른지
- **첫 번째 합성곱 층**과 **마지막 합성곱 층**의 Grad-CAM 히트맵이 어떻게 다른지

를 비교합니다. 다운로드 없이 CPU 에서 몇 초면 끝나요. (필요: `pip install torch`)

> 💡 PyRewind 로 실행하면 히트맵이 **색**으로 나오고, 같은 크기의 사진 변수(`x`) 위에 **겹쳐서**도 보여줍니다.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0);   # 끝의 ; 는 주피터처럼 값 출력을 숨김

## 1. 데이터 만들기 — 원 또는 네모가 아무 곳에나 있는 그림
`os.environ.get("PYREWIND")` 는 **PyRewind 로 실행할 때만** 값이 있어요. 시각화할 때는 데이터를 작게 써서 빨리 끝나게 합니다.
(큰 데이터셋 / GPU 학습 코드도 이렇게 해 두면 PyRewind 로 흐름만 빠르게 확인할 수 있어요.)

In [ ]:
N = 64 if os.environ.get("PYREWIND") else 256
SIZE = 32

kinds = torch.randint(0, 2, (N,))                       # 0 = 원, 1 = 네모
cx = torch.randint(8, SIZE - 8, (N, 1, 1))              # 도형 중심 위치
cy = torch.randint(8, SIZE - 8, (N, 1, 1))
r = torch.randint(4, 7, (N, 1, 1))                      # 크기
yy, xx = torch.meshgrid(torch.arange(SIZE), torch.arange(SIZE), indexing="ij")
circle = (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2
square = ((xx - cx).abs() <= r) & ((yy - cy).abs() <= r)
mask = torch.where(kinds[:, None, None] == 0, circle, square)
colors = torch.rand(N, 3, 1, 1) * 0.6 + 0.4
images = mask[:, None] * colors + 0.05 * torch.randn(N, 3, SIZE, SIZE)
images.shape

## 2. 작은 CNN — 합성곱 층 3개

In [ ]:
model = nn.Sequential(
    nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(),                       # [0] 첫 번째 합성곱
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # [2] 두 번째 (크기 절반)
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),                     # [5] 마지막 합성곱
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 2),
)
model

## 3. 학습
반복문이 길면 VS Code 설정 **`pyrewind.loopLimit`** (예: `3`) 을 켜 보세요. 처음 3번만 한 줄씩 기록하고 나머지는 빠르게 실행합니다.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
losses = []
for epoch in range(60):
    logits = model(images)
    loss = F.cross_entropy(logits, kinds)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(round(loss.item(), 4))

accuracy = (model(images).argmax(dim=1) == kinds).float().mean().item()
accuracy

## 4. 그림 한 장을 골라 층별 특징 맵 보기
슬라이더로 채널을 넘겨 보세요. 첫 층은 **선 / 경계 / 색** 처럼 단순한 것에, 마지막 층은 **도형 전체** 에 반응해요. (마지막 층은 크기가 절반)

In [ ]:
x = images[:1]                       # (1, 3, 32, 32) 사진 한 장
label = int(kinds[0])

feat_first = model[1](model[0](x))   # 첫 번째 합성곱 + ReLU → (1, 8, 32, 32)
feat_last = model[:7](x)             # 마지막 합성곱 + ReLU  → (1, 32, 16, 16)

## 5. Grad-CAM 함수
1. 원하는 층에 **hook** 을 달아 그 층의 출력(activation)과 기울기(gradient)를 저장
2. 정답 클래스 점수를 거꾸로 미분 (`backward`)
3. 채널마다 기울기 평균 = **그 채널이 얼마나 중요한지** → 가중합 → ReLU → 그림 크기로 키우기

함수 안으로 들어가면 hook 이 실행되는 순간도 한 줄씩 볼 수 있어요.

In [ ]:
def grad_cam(layer, x, target):
    saved = {}

    def keep(module, inputs, output):          # 순전파 때 이 층의 출력을 저장
        saved["act"] = output
        output.register_hook(lambda g: saved.update(grad=g))   # 역전파 때 기울기 저장

    handle = layer.register_forward_hook(keep)
    out = model(x)
    model.zero_grad()
    out[0, target].backward()
    handle.remove()

    weights = saved["grad"].mean(dim=(2, 3), keepdim=True)          # 채널별 중요도
    cam = F.relu((weights * saved["act"]).sum(dim=1, keepdim=True))  # 가중합
    cam = F.interpolate(cam, size=x.shape[2:], mode="bilinear", align_corners=False)
    return (cam / (cam.max() + 1e-8))[0, 0].detach()                # 0~1 로 맞춤

## 6. 첫 번째 층 vs 마지막 층
- `cam_first` : 첫 합성곱 — 그림 여기저기 (경계, 배경 잡음) 에 흩어져 있어요
- `cam_last`  : 마지막 합성곱 — **도형이 있는 자리**에 모여 있어요

변수 칸에서 두 히트맵과 `x` 위에 겹친 그림을 비교해 보세요.

In [ ]:
cam_first = grad_cam(model[0], x, label)
cam_last = grad_cam(model[5], x, label)

# 도형 안 / 밖의 평균 히트맵 값으로 비교 (마지막 층은 안쪽이 훨씬 큼)
inside = mask[0].float()
first_in_out = (round((cam_first * inside).sum().item() / inside.sum().item(), 3),
                round((cam_first * (1 - inside)).sum().item() / (1 - inside).sum().item(), 3))
last_in_out = (round((cam_last * inside).sum().item() / inside.sum().item(), 3),
               round((cam_last * (1 - inside)).sum().item() / (1 - inside).sum().item(), 3))
first_in_out, last_in_out